# Operaciones con Azure Storage

Este notebook contiene métodos para trabajar con Azure Blob Storage:
- **Descargar archivos** desde un contenedor
- **Subir archivos** a un contenedor


## Instalación de dependencias

Primero, instalamos la biblioteca necesaria para trabajar con Azure Storage.


In [10]:
# Instalar azure-storage-blob si no está instalado
!pip install azure-storage-blob


## Importación de librerías


In [11]:
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import AzureError
import os
from typing import Optional


## Método 1: Descargar archivo de Azure Storage

Este método conecta a Azure Storage, accede al contenedor especificado y descarga el archivo indicado.


In [12]:
def descargar_archivo_azure(
    connection_string: str,
    nombre_contenedor: str,
    nombre_archivo: str,
    ruta_destino: Optional[str] = None
) -> str:
    """
    Descarga un archivo desde Azure Blob Storage.
    
    Parámetros:
    -----------
    connection_string : str
        Cadena de conexión de Azure Storage Account
    nombre_contenedor : str
        Nombre del contenedor en Azure Storage
    nombre_archivo : str
        Nombre del archivo (blob) a descargar
    ruta_destino : str, opcional
        Ruta local donde guardar el archivo. Si no se especifica,
        se guarda en el directorio actual con el mismo nombre.
    
    Retorna:
    --------
    str
        Ruta completa del archivo descargado
    
    Ejemplo:
    --------
    >>> archivo = descargar_archivo_azure(
    ...     connection_string="DefaultEndpointsProtocol=https;AccountName=...",
    ...     nombre_contenedor="mi-contenedor",
    ...     nombre_archivo="documento.pdf",
    ...     ruta_destino="./descargas/documento.pdf"
    ... )
    """
    try:
        # Crear cliente de Blob Service
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        
        # Obtener cliente del contenedor
        container_client = blob_service_client.get_container_client(nombre_contenedor)
        
        # Verificar que el contenedor existe
        if not container_client.exists():
            raise ValueError(f"El contenedor '{nombre_contenedor}' no existe")
        
        # Obtener cliente del blob
        blob_client = container_client.get_blob_client(nombre_archivo)
        
        # Verificar que el archivo existe
        if not blob_client.exists():
            raise ValueError(f"El archivo '{nombre_archivo}' no existe en el contenedor")
        
        # Determinar ruta de destino
        if ruta_destino is None:
            ruta_destino = nombre_archivo
        
        # Crear directorio si no existe
        directorio = os.path.dirname(ruta_destino)
        if directorio and not os.path.exists(directorio):
            os.makedirs(directorio, exist_ok=True)
        
        # Descargar el archivo
        print(f"Descargando '{nombre_archivo}' desde el contenedor '{nombre_contenedor}'...")
        with open(ruta_destino, "wb") as archivo_local:
            datos = blob_client.download_blob()
            archivo_local.write(datos.readall())
        
        print(f"Archivo descargado exitosamente en: {os.path.abspath(ruta_destino)}")
        return os.path.abspath(ruta_destino)
        
    except AzureError as e:
        print(f"Error de Azure Storage: {e}")
        raise
    except Exception as e:
        print(f"Error inesperado: {e}")
        raise


## Método 2: Subir archivo a Azure Storage

Este método conecta a Azure Storage, accede al contenedor especificado y sube el archivo indicado.


In [13]:
def subir_archivo_azure(
    connection_string: str,
    nombre_contenedor: str,
    ruta_archivo_local: str,
    nombre_archivo_destino: Optional[str] = None,
    sobrescribir: bool = True
) -> str:
    """
    Sube un archivo a Azure Blob Storage.
    
    Parámetros:
    -----------
    connection_string : str
        Cadena de conexión de Azure Storage Account
    nombre_contenedor : str
        Nombre del contenedor en Azure Storage
    ruta_archivo_local : str
        Ruta completa del archivo local a subir
    nombre_archivo_destino : str, opcional
        Nombre que tendrá el archivo en Azure Storage.
        Si no se especifica, se usa el nombre del archivo local.
    sobrescribir : bool, opcional
        Si es True, sobrescribe el archivo si ya existe.
        Si es False, lanza un error si el archivo ya existe.
        Por defecto es True.
    
    Retorna:
    --------
    str
        URL del archivo subido en Azure Storage
    
    Ejemplo:
    --------
    >>> url = subir_archivo_azure(
    ...     connection_string="DefaultEndpointsProtocol=https;AccountName=...",
    ...     nombre_contenedor="mi-contenedor",
    ...     ruta_archivo_local="./documentos/informe.pdf",
    ...     nombre_archivo_destino="informes/informe_2024.pdf"
    ... )
    """
    try:
        # Verificar que el archivo local existe
        if not os.path.exists(ruta_archivo_local):
            raise FileNotFoundError(f"El archivo '{ruta_archivo_local}' no existe")
        
        # Crear cliente de Blob Service
        blob_service_client = BlobServiceClient.from_connection_string(connection_string)
        
        # Obtener cliente del contenedor
        container_client = blob_service_client.get_container_client(nombre_contenedor)
        
        # Crear contenedor si no existe
        if not container_client.exists():
            print(f"Creando contenedor '{nombre_contenedor}'...")
            container_client.create_container()
            print(f"Contenedor '{nombre_contenedor}' creado exitosamente")
        
        # Determinar nombre del archivo destino
        if nombre_archivo_destino is None:
            nombre_archivo_destino = os.path.basename(ruta_archivo_local)
        
        # Obtener cliente del blob
        blob_client = container_client.get_blob_client(nombre_archivo_destino)
        
        # Verificar si el archivo ya existe
        if blob_client.exists() and not sobrescribir:
            raise ValueError(
                f"El archivo '{nombre_archivo_destino}' ya existe en el contenedor. "
                "Use sobrescribir=True para reemplazarlo."
            )
        
        # Subir el archivo
        print(f"Subiendo '{ruta_archivo_local}' al contenedor '{nombre_contenedor}' como '{nombre_archivo_destino}'...")
        with open(ruta_archivo_local, "rb") as archivo_local:
            blob_client.upload_blob(archivo_local.read(), overwrite=sobrescribir)
        
        # Obtener URL del archivo subido
        url_archivo = blob_client.url
        print(f"Archivo subido exitosamente. URL: {url_archivo}")
        return url_archivo
        
    except AzureError as e:
        print(f"Error de Azure Storage: {e}")
        raise
    except Exception as e:
        print(f"Error inesperado: {e}")
        raise


## Ejemplos de uso

A continuación se muestran ejemplos de cómo usar ambos métodos.


### Configuración

Primero, configura tu cadena de conexión de Azure Storage. Puedes obtenerla desde Azure Portal.


In [14]:
# Configura tu cadena de conexión de Azure Storage
# Puedes obtenerla desde Azure Portal > Storage Account > Access Keys
CONNECTION_STRING = "#"

# Nombre del contenedor
NOMBRE_CONTENEDOR = "#"


### Ejemplo 1: Subir un archivo


In [15]:
# Ejemplo: Subir un archivo
ruta_local = "./archivo-subida.txt"
url_subida = subir_archivo_azure(
     connection_string=CONNECTION_STRING,
     nombre_contenedor=NOMBRE_CONTENEDOR,
     ruta_archivo_local=ruta_local,
     nombre_archivo_destino="archivos/mi_archivo.txt"
    )
print(f"Archivo disponible en: {url_subida}")


Creando contenedor 'neurofinder-contenedor'...
Contenedor 'neurofinder-contenedor' creado exitosamente
Subiendo './archivo-subida.txt' al contenedor 'neurofinder-contenedor' como 'archivos/mi_archivo.txt'...
Archivo subido exitosamente. URL: https://neurofinder.blob.core.windows.net/neurofinder-contenedor/archivos/mi_archivo.txt
Archivo disponible en: https://neurofinder.blob.core.windows.net/neurofinder-contenedor/archivos/mi_archivo.txt


### Ejemplo 2: Descargar un archivo


In [16]:
# Ejemplo: Descargar un archivo
nombre_archivo_azure = "archivos/mi_archivo.txt"

ruta_descarga = descargar_archivo_azure(
     connection_string=CONNECTION_STRING,
     nombre_contenedor=NOMBRE_CONTENEDOR,
     nombre_archivo=nombre_archivo_azure,
     ruta_destino="./descargas/mi_archivo_descargado.txt"
)

print(f"Archivo descargado en: {ruta_descarga}")


Descargando 'archivos/mi_archivo.txt' desde el contenedor 'neurofinder-contenedor'...
Archivo descargado exitosamente en: /Users/josevicenteantoncoy/Documents/Git TFM/Pruebas/descargas/mi_archivo_descargado.txt
Archivo descargado en: /Users/josevicenteantoncoy/Documents/Git TFM/Pruebas/descargas/mi_archivo_descargado.txt
